# 실기 대비
# 실전 문제풀이
# set 5

## 1) 데이터 및 시나리오

### 신용카드 고객정보 분석

> 삼성전자의 고객 마케팅에 활용하기 위하여 삼성 카드 측에서 비식별화된 고객 데이터를 협조 받아 통합 분석을 하기 위한 선제 분석을 하고자 한다.

※ 분석 수행 전 `기한 내 최소 지불 금액(MINIMUM_PAYMENTS)`의 결측 값(Null)을 각 컬럼의 평균값으로 대체하시오.

전처리 수행 결과를 `base` 객체로 지정하고 다음 문항에서 해당 객체를 기반으로 문제를 풀이하시오.

### 데이터 개요

| 파일명 | 행 | 열 | 인코딩 |
|---|---:|---:|---|
| `card_cust.csv` | 1000 | 18 | UTF-8 |

## 1) 데이터 및 시나리오

### 변수 상세

| 변수명 | 유형 | 설명 |
|---|---|---|
| `CUST_ID` | int | 고객 ID |
| `BALANCE` | float | 연간 평균 잔고액 |
| `BALANCE_FREQUENCY` | float | 연중 잔고액 갱신 개월 수 비율 `(0~1 사이값)` |
| `PURCHASES` | float | 구매 총액 |
| `ONEOFF_PURCHASES` | float | 일시불 구매 총액 |
| `INSTALLMENTS_PURCHASES` | float | 할부 구매 총액 |
| `CASH_ADVANCE` | float | 현금서비스 구매 총액 |
| `PURCHASES_FREQUENCY` | float | 연중 구매 개월 수 비율 `(0~1 사이값)` |
| `ONEOFF_PURCHASES_FREQUENCY` | float | 연중 일시불 구매 개월 수 비율 `(0~1 사이값)` |
| `PURCHASES_INSTALLMENTS_FREQUENCY` | float | 연중 할부 구매 개월 수 비율 `(0~1 사이값)` |
| `CASH_ADVANCE_FREQUENCY` | float | 연중 현금서비스 구매 개월 수 비율 |
| `CASH_ADVANCE_TRX` | int | 현금 서비스 구매 횟수 |
| `PURCHASES_TRX` | int | 구매 횟수 |
| `CREDIT_LIMIT` | int | 신용카드 한도 |
| `PAYMENTS` | float | 지불 총액 |
| `MINIMUM_PAYMENTS` | float | 기한 내 최소 지불 금액 |
| `PRC_FULL_PAYMENT` | float | 연중 기한 내 전액 지불 개월 수 비율 `(0~1 사이값)` |
| `TENURE` | float | 신용카드 서비스 이용기간 |

## 2) 문제

### 필요 라이브러리 함수 및 클래스 목록

| 목록 |
|---|
| `from sklearn.preprocessing import StandardScaler` |
| `from sklearn.cluster import KMeans` |
| `from sklearn.metrics import silhouette_score` |
| `from sklearn.tree import DecisionTreeRegressor` |


In [4]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.tree import DecisionTreeRegressor

df = pd.read_csv('../dataset/card_cust.csv')
display(df.shape)
display(df.columns)
display(df)

(1000, 18)

Index(['CUST_ID', 'BALANCE', 'BALANCE_FREQUENCY', 'PURCHASES',
       'ONEOFF_PURCHASES', 'INSTALLMENTS_PURCHASES', 'CASH_ADVANCE',
       'PURCHASES_FREQUENCY', 'ONEOFF_PURCHASES_FREQUENCY',
       'PURCHASES_INSTALLMENTS_FREQUENCY', 'CASH_ADVANCE_FREQUENCY',
       'CASH_ADVANCE_TRX', 'PURCHASES_TRX', 'CREDIT_LIMIT', 'PAYMENTS',
       'MINIMUM_PAYMENTS', 'PRC_FULL_PAYMENT', 'TENURE'],
      dtype='object')

,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,10001,40.900749,0.818182,95.40,0.00,95.40,0.000000,0.166667,0.000000,0.083333,0.000000,0.0,2.0,1000.0,201.802084,139.509787,0.000000,12.0
1,10002,3202.467416,0.909091,0.00,0.00,0.00,6442.945483,0.000000,0.000000,0.000000,0.250000,4.0,0.0,7000.0,4103.032597,1072.340217,0.222222,12.0
2,10003,2495.148862,1.000000,773.17,773.17,0.00,0.000000,1.000000,1.000000,0.000000,0.000000,0.0,12.0,7500.0,622.066742,627.284787,0.000000,12.0
3,10004,1666.670542,0.636364,1499.00,1499.00,0.00,205.788017,0.083333,0.083333,0.000000,0.083333,1.0,1.0,7500.0,0.000000,NaN,0.000000,12.0
4,10005,817.714335,1.000000,16.00,16.00,0.00,0.000000,0.083333,0.083333,0.000000,0.000000,0.0,1.0,1200.0,678.334763,244.791237,0.000000,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,11029,1250.394614,0.909091,443.99,443.99,0.00,0.000000,0.272727,0.272727,0.000000,0.000000,0.0,3.0,3500.0,273.823646,259.715939,0.000000,11.0
996,11030,9.503968,1.000000,96.62,0.00,96.62,0.000000,1.000000,0.000000,1.000000,0.000000,0.0,19.0,4500.0,1086.932525,92.217936,0.250000,12.0
997,11031,2285.068731,1.000000,0.00,0.00,0.00,1173.310874,0.000000,0.000000,0.000000,0.166667,5.0,0.0,2500.0,381.672065,1003.265207,0.000000,12.0
998,11032,2928.756699,1.000000,160.92,0.00,160.92,319.931964,1.000000,0.000000,1.000000,0.333333,5.0,12.0,3000.0,1142.847203,1098.479834,0.000000,12.0


In [8]:
# 전처리
df_base = df.copy()
display(df['MINIMUM_PAYMENTS'].isna().sum())

df_base['MINIMUM_PAYMENTS'] = df_base['MINIMUM_PAYMENTS'].fillna(df_base['MINIMUM_PAYMENTS'].mean())
display(df_base['MINIMUM_PAYMENTS'].isna().sum())

74

0

### Q01.

`base`를 사용하여 연간 평균 잔고액과 신용카드 서비스 이용기간 간의 관계를 파악하여, 추후 고객의 신용카드 한도 조정에 근거 자료로 활용하고자 한다.

연간 평균 잔고액(`BALANCE`)이 많을수록, 그리고 신용카드 서비스 이용기간(`TENURE`)이 길수록 신용카드 한도(`CREDIT_LIMIT`) 역시 높을 것으로 예상해볼 수 있다. 신용 카드 서비스 이용기간(`TENURE`) 별로 연간 평균 잔고액(`BALANCE`)과 신용카드 한도(`CREDIT_LIMIT`) 간 피어슨(Pearson) 상관 분석을 실시하고, 이 중 가장 큰 상관계수를 구하시오.

※ 정답은 반올림하여 소수점 둘째 자리까지 출력하시오. `(정답 예시: 0.12)`

In [19]:
df_q1 = df_base.copy()
#display(df_q1.groupby(['TENURE'])['BALANCE','CREDIT_LIMIT'].corr(method='pearson'))
df_q1_gb = df_q1.groupby('TENURE')['BALANCE','CREDIT_LIMIT'].apply(
    lambda df : df['BALANCE'].corr(df['CREDIT_LIMIT'])
    ) #상세설명
display(df_q1_gb)

round(df_q1_gb.max(), 2)

x:\study\docs\data_science\.venv\lib\site-packages\ipykernel_launcher.py:3: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  This is separate from the ipykernel package so we can avoid doing imports until


TENURE
6.0     0.868056
7.0     0.948405
8.0     0.820696
9.0     0.085474
10.0    0.291482
11.0    0.380360
12.0    0.460833
dtype: float64

0.95

### Q02.

`base`를 사용하여 전략을 수립하기 위해 고객 세분화를 수행하고자 한다.  
일시불 구매 금액이 높은 고객군을 도출하기 위해 다음 단계에 따라 분석을 수행하고 질문에 답하시오.

#### 단계 1

`고객 ID`를 제외한 모든 변수 17개에 대해 Z-score 표준화(Standardization) 한다.

#### 단계 2

표준화된 변수들에 대해 K-means 군집 분석을 수행한다.

이 때, 군집 수는 2~5개 중 K-means Silhouette를 통해 구한 최적의 K로 설정한다.

#### 단계 3

단계 2에서 도출한 각 군집 별로 `일시불 구매 총액`의 평균을 계산한다.

**군집 별 일시불 구매 총액(`ONEOFF_PURCHASES`)의 평균 중 가장 큰 값은 얼마인가?**

※ 정규화를 실시하지 않은 일시불 구매 총액 데이터를 기준으로 평균을 산출하시오.  
※ seed는 `1234`로 설정하시오.  
※ 정답은 반올림하여 소수점 둘째 자리까지 출력하시오. `(정답 예시: 0.12)`

In [51]:
df_q2 = df_base.copy()
#단계 1
#D
X = df_q2.drop(columns=['CUST_ID']).copy()
#N
scaler = StandardScaler()
X_n = scaler.fit_transform(X) #array
display(X_n)
#단계 2
label = {}
sil = {}
for k in range(2,6) :
    model = KMeans(n_clusters = k,
                   random_state = 1234)
    y_pred = model.fit_predict(X_n)
    label[k] = y_pred
    sil[k] = silhouette_score(X_n,y_pred)

ser_sil = pd.Series(sil, name = 'sil')
K = ser_sil.idxmax()
display(K, ser_sil)
display("------------------")
df_label = pd.DataFrame(label[K])
display(df_label)

#단계 3
#X['cluster'] = df_label # series를 바로 붙인다.
X['cluster'] = label[K] # dict을 바로 붙인다.
X_gb = X.groupby('cluster')['ONEOFF_PURCHASES'].mean()
display(X_gb.idxmax(), X_gb.max())
round(X_gb.max(),2)


array([[-0.84876759, -0.41987944, -0.4419358 , ..., -0.44372465,
        -0.46554357,  0.28242902],
       [ 0.28279099,  0.01213096, -0.4690169 , ..., -0.08615941,
         0.33159169,  0.28242902],
       [ 0.0296341 ,  0.44414137, -0.24953794, ..., -0.25675456,
        -0.46554357,  0.28242902],
       ...,
       [-0.04555583,  0.44414137, -0.4690169 , ..., -0.1126367 ,
        -0.46554357,  0.28242902],
       [ 0.18482699,  0.44414137, -0.4233367 , ..., -0.07613978,
        -0.46554357,  0.28242902],
       [-0.34079285,  0.44414137, -0.20275918, ...,  0.00909944,
        -0.46554357,  0.28242902]])

2

2    0.307528
3    0.196361
4    0.207151
5    0.192741
Name: sil, dtype: float64

'------------------'

,0
0,0
1,0
2,0
3,0
4,0
...,...
995,0
996,0
997,0
998,0


1

3946.187525252525

3946.19

### Q03.

`base`를 사용하여 일시불 구매 총액(`ONEOFF_PURCHASES`) 예측 모델을 Target Marketing에 활용하고자 한다. 다음 단계에 따라 분석을 수행하고 질문에 답하시오.

#### 단계 1

`고객 ID(CUST_ID)`가 4의 배수가 아닌 데이터를 Train Set으로, 4의 배수인 데이터를 Test Set으로 분할한다.

#### 단계 2

Train Set으로 아래 조건에 따라 의사결정나무 회귀모델을 학습한다.

- 독립 변수 총 16개: `고객 ID`, `일시불 구매 총액`을 제외한 모든 변수
- 종속 변수: `일시불 구매 총액`

#### 단계 3

생성된 모델을 Test Set에 적용하여 `일시불 구매 총액`을 예측한다.

**단계 3에서 얻은 예측 결과를 평가하기 위해, 아래 정의된 Measure B를 계산한 값은?**

$$
B = \left( \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y_i})^2 \right)^{\frac{1}{2}}
$$

- $\hat{y_i}$: 예측값
- $y_i$: 실제값

※ seed는 `1234`로 설정하시오.  
※ 정답은 반올림하여 소수점 둘째 자리까지 출력하시오. `(정답 예시: 0.12)`

In [50]:
df_q3 = df_base.copy()

#단계1
train = df_q3.loc[df_q3['CUST_ID']%4 != 0, :].copy()
test = df_q3.loc[df_q3['CUST_ID']%4 == 0, :].copy()
display(train.shape, test.shape)
#단계2
#D
X_train = train.drop(columns = ['CUST_ID', 'ONEOFF_PURCHASES']).copy()
X_test = test.drop(columns = ['CUST_ID', 'ONEOFF_PURCHASES']).copy()

y_train = train['ONEOFF_PURCHASES'].copy()
y_test = test['ONEOFF_PURCHASES'].copy()

#N
#M
model = DecisionTreeRegressor(random_state = 1234)
model.fit(X = X_train,y = y_train)
y_pred = model.predict(X_test)
#단계3
#E
B = ((y_pred - y_test)**2).mean()**0.5
round(B, 2)

(752, 18)

(248, 18)

1039.19

---

# Q1 상세 설명
```
df_q1_gb = df_q1.groupby('TENURE')['BALANCE','CREDIT_LIMIT'].apply(
    lambda df : df['BALANCE'].corr(df['CREDIT_LIMIT'])
    )
```
좋아. 이 코드는 **객체가 어떻게 변하면서 실행되는지**로 보면 훨씬 쉬워져.

먼저 최종 코드는 이렇게 쓰는 게 좋아.

```python id="m0ol7h"
df_q1_gb = df_q1.groupby('TENURE')[['BALANCE', 'CREDIT_LIMIT']].apply(
    lambda df: df['BALANCE'].corr(df['CREDIT_LIMIT'])
)
```

---

# 전체 코드 구조

```python id="5nxy3f"
df_q1
.groupby('TENURE')
[['BALANCE', 'CREDIT_LIMIT']]
.apply(lambda df: df['BALANCE'].corr(df['CREDIT_LIMIT']))
```

이걸 객체 흐름으로 보면:

```text id="5ki45l"
DataFrame
   ↓
DataFrameGroupBy
   ↓
DataFrameGroupBy에서 필요한 컬럼만 선택
   ↓
각 그룹이 작은 DataFrame으로 lambda에 들어감
   ↓
각 그룹마다 corr 숫자 1개 반환
   ↓
최종 결과는 Series
```

---

# 1단계. 원본 객체 `df_q1`

처음 `df_q1`은 그냥 DataFrame이야.

```python id="8e3455"
type(df_q1)
```

결과:

```text id="z04qwd"
pandas.core.frame.DataFrame
```

예시 모양:

```text id="bt7h7y"
df_q1

┌───────┬────────┬─────────┬──────────────┐
│ index │ TENURE │ BALANCE │ CREDIT_LIMIT │
├───────┼────────┼─────────┼──────────────┤
│   0   │   6    │   100   │    1000      │
│   1   │   6    │   200   │    1500      │
│   2   │   6    │   300   │    2000      │
│   3   │  12    │   500   │    3000      │
│   4   │  12    │   700   │    3200      │
│   5   │  12    │   900   │    4000      │
└───────┴────────┴─────────┴──────────────┘
```

---

# 2단계. `groupby('TENURE')`

```python id="1yh5o5"
gb = df_q1.groupby('TENURE')
```

이때 바로 표가 만들어지는 게 아니라, **TENURE별로 나눌 준비가 된 객체**가 만들어져.

```python id="thtbx3"
type(gb)
```

결과:

```text id="hbsld7"
pandas.core.groupby.generic.DataFrameGroupBy
```

객체 모양은 이렇게 생각하면 돼.

```text id="dwu8k0"
gb = df_q1.groupby('TENURE')

DataFrameGroupBy 객체

TENURE = 6  → index [0, 1, 2]
TENURE = 12 → index [3, 4, 5]
```

즉 아직 결과값이 아니라:

```text id="r5krhu"
"나중에 계산할 때 TENURE별로 쪼개줄게"
```

라는 상태야.

---

# 3단계. `[['BALANCE', 'CREDIT_LIMIT']]`

```python id="5ynrtv"
gb_selected = df_q1.groupby('TENURE')[['BALANCE', 'CREDIT_LIMIT']]
```

이것도 아직 결과표가 아니야.
**그룹별로 나눈 뒤, 각 그룹에서 사용할 컬럼을 `BALANCE`, `CREDIT_LIMIT`로 제한한 GroupBy 객체**야.

```python id="z2abkj"
type(gb_selected)
```

결과:

```text id="mv5byn"
pandas.core.groupby.generic.DataFrameGroupBy
```

객체 구조는 이렇게 바뀐다고 보면 돼.

```text id="plj55h"
gb_selected

TENURE = 6 그룹에서 사용할 컬럼
┌───────┬─────────┬──────────────┐
│ index │ BALANCE │ CREDIT_LIMIT │
├───────┼─────────┼──────────────┤
│   0   │   100   │    1000      │
│   1   │   200   │    1500      │
│   2   │   300   │    2000      │
└───────┴─────────┴──────────────┘

TENURE = 12 그룹에서 사용할 컬럼
┌───────┬─────────┬──────────────┐
│ index │ BALANCE │ CREDIT_LIMIT │
├───────┼─────────┼──────────────┤
│   3   │   500   │    3000      │
│   4   │   700   │    3200      │
│   5   │   900   │    4000      │
└───────┴─────────┴──────────────┘
```

여기서 중요한 점.

`lambda df` 안으로 들어가는 `df`는 원본 `df_q1` 전체가 아니라,
**TENURE별로 잘린 작은 DataFrame**이야.

---

# 4단계. `apply(lambda df: ...)`

```python id="2gek0o"
.apply(
    lambda df: df['BALANCE'].corr(df['CREDIT_LIMIT'])
)
```

이제 진짜 계산이 시작돼.

---

## 첫 번째 반복: TENURE = 6

`apply()`가 첫 번째 그룹을 `lambda df` 안에 넣어.

```python id="ptn58i"
lambda df: df['BALANCE'].corr(df['CREDIT_LIMIT'])
```

이때 `df`의 실제 모양:

```text id="35nain"
df   # TENURE = 6 그룹

┌───────┬─────────┬──────────────┐
│ index │ BALANCE │ CREDIT_LIMIT │
├───────┼─────────┼──────────────┤
│   0   │   100   │    1000      │
│   1   │   200   │    1500      │
│   2   │   300   │    2000      │
└───────┴─────────┴──────────────┘
```

여기서:

```python id="g547wb"
df['BALANCE']
```

는 Series야.

```text id="qy50nb"
index
0    100
1    200
2    300
Name: BALANCE, dtype: int64
```

그리고:

```python id="s8piwl"
df['CREDIT_LIMIT']
```

도 Series야.

```text id="oapjdm"
index
0    1000
1    1500
2    2000
Name: CREDIT_LIMIT, dtype: int64
```

그래서 이 계산이 실행돼.

```python id="afzouu"
df['BALANCE'].corr(df['CREDIT_LIMIT'])
```

결과:

```text id="i5hajn"
1.0
```

즉 첫 번째 그룹의 반환값은:

```text id="vt7o5s"
TENURE = 6 → 1.0
```

---

## 두 번째 반복: TENURE = 12

이번에는 `TENURE = 12` 그룹이 `lambda df`로 들어가.

이때 `df`의 실제 모양:

```text id="14wb3o"
df   # TENURE = 12 그룹

┌───────┬─────────┬──────────────┐
│ index │ BALANCE │ CREDIT_LIMIT │
├───────┼─────────┼──────────────┤
│   3   │   500   │    3000      │
│   4   │   700   │    3200      │
│   5   │   900   │    4000      │
└───────┴─────────┴──────────────┘
```

내부에서는 똑같이:

```python id="lnp1dh"
df['BALANCE'].corr(df['CREDIT_LIMIT'])
```

이 실행돼.

```text id="pzsm75"
BALANCE Series
3    500
4    700
5    900

CREDIT_LIMIT Series
3    3000
4    3200
5    4000
```

결과 예시:

```text id="s495ps"
0.94
```

즉 두 번째 그룹의 반환값은:

```text id="qeb2xr"
TENURE = 12 → 0.94
```

---

# 5단계. apply가 반환값들을 모아서 Series 생성

각 그룹에서 숫자 1개씩 나왔지?

```text id="zcod9k"
TENURE = 6  → 1.00
TENURE = 12 → 0.94
```

`apply()`는 이걸 모아서 최종적으로 **Series**를 만들어.

```text id="3i0lb4"
df_q1_gb

TENURE
6     1.00
12    0.94
dtype: float64
```

객체 타입:

```python id="v1zxry"
type(df_q1_gb)
```

결과:

```text id="rgl67e"
pandas.core.series.Series
```

---

# 전체 객체 변화 최종 그림

```text id="q2cw5o"
df_q1
DataFrame
┌────────┬─────────┬──────────────┐
│ TENURE │ BALANCE │ CREDIT_LIMIT │
└────────┴─────────┴──────────────┘
        │
        │ groupby('TENURE')
        ▼
gb
DataFrameGroupBy
{
  6  : rows [0, 1, 2],
  12 : rows [3, 4, 5]
}
        │
        │ [['BALANCE', 'CREDIT_LIMIT']]
        ▼
gb_selected
DataFrameGroupBy
{
  6  : DataFrame[['BALANCE', 'CREDIT_LIMIT']],
  12 : DataFrame[['BALANCE', 'CREDIT_LIMIT']]
}
        │
        │ apply(lambda df: ...)
        ▼
각 그룹이 df로 들어감

TENURE=6의 df
┌─────────┬──────────────┐
│ BALANCE │ CREDIT_LIMIT │
└─────────┴──────────────┘
        ↓
corr 계산
        ↓
1.00

TENURE=12의 df
┌─────────┬──────────────┐
│ BALANCE │ CREDIT_LIMIT │
└─────────┴──────────────┘
        ↓
corr 계산
        ↓
0.94
        │
        ▼
최종 결과 Series

TENURE
6     1.00
12    0.94
dtype: float64
```

---

# 코드 설명 완성본

```python id="lq5w43"
df_q1_gb = df_q1.groupby('TENURE')[['BALANCE', 'CREDIT_LIMIT']].apply(
    lambda df: df['BALANCE'].corr(df['CREDIT_LIMIT'])
)
```

## 해석

```python id="2z0kgq"
df_q1.groupby('TENURE')
```

`TENURE` 값별로 데이터를 그룹화한다.

```python id="k6skkf"
[['BALANCE', 'CREDIT_LIMIT']]
```

각 그룹에서 사용할 컬럼을 `BALANCE`, `CREDIT_LIMIT` 두 개로 제한한다.

```python id="ewkvjj"
.apply(...)
```

그룹별로 같은 계산을 반복한다.

```python id="d1ot14"
lambda df: df['BALANCE'].corr(df['CREDIT_LIMIT'])
```

각 그룹 안에서 `BALANCE`와 `CREDIT_LIMIT`의 상관계수를 계산한다.

---

# 결과를 보기 좋게 만들기

현재 결과는 Series야.

```text id="7fnynh"
TENURE
6     1.00
12    0.94
dtype: float64
```

표처럼 보고 싶으면:

```python id="balbwu"
df_q1_gb = df_q1_gb.reset_index(name='corr')
display(df_q1_gb)
```

결과:

```text id="1so1pn"
┌────────┬──────┐
│ TENURE │ corr │
├────────┼──────┤
│   6    │ 1.00 │
│  12    │ 0.94 │
└────────┴──────┘
```

---

# 진짜 핵심 한 문장

```text id="gks3om"
lambda df의 df는 원본 전체 데이터가 아니라, groupby로 TENURE별로 잘린 작은 DataFrame이다.
```

그래서 이 코드는 결국:

```text id="1t2pix"
TENURE별로 BALANCE와 CREDIT_LIMIT의 상관계수를 각각 구하는 코드
```

야.
|